In [15]:
import pandas as pd
import plotly.express as px

TARGET_PATH = './data/data.csv'

df = pd.read_csv(TARGET_PATH, encoding='utf-8')

df.head()

,출발항코드(DEPARTURE_PORT_CODE),출발항명(DEPARTURE_PORT_NAME),도착항코드(DEST_PORT_CODE),도착항명(DEST_PORT_NAME),도착항국가(DEST_COUNTRY),기준일자(DATE),항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
0,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-05-28,23.69,644.50,0.0,7.94
1,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,22.01,608.54,0.0,16.20
2,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,16.52,749.45,0.0,1.36
3,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,22.37,634.28,0.0,7.44
4,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,16.37,654.99,0.0,1.80


In [16]:
df.shape

(480, 10)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   출발항코드(DEPARTURE_PORT_CODE)  480 non-null    str    
 1   출발항명(DEPARTURE_PORT_NAME)   480 non-null    str    
 2   도착항코드(DEST_PORT_CODE)       480 non-null    str    
 3   도착항명(DEST_PORT_NAME)        480 non-null    str    
 4   도착항국가(DEST_COUNTRY)         426 non-null    str    
 5   기준일자(DATE)                  480 non-null    str    
 6   항만효율성(PORT_EFFICIENCY)      480 non-null    float64
 7   총항해시간(TOTAL_SAILING_TIME)   480 non-null    float64
 8   대기시간(WAITING_TIME)          480 non-null    float64
 9   항만정시성(ON_TIME_PERFORMANCE)  480 non-null    float64
dtypes: float64(4), str(6)
memory usage: 59.8 KB


In [18]:
df.isnull().sum()

출발항코드(DEPARTURE_PORT_CODE)     0
출발항명(DEPARTURE_PORT_NAME)      0
도착항코드(DEST_PORT_CODE)          0
도착항명(DEST_PORT_NAME)           0
도착항국가(DEST_COUNTRY)           54
기준일자(DATE)                     0
항만효율성(PORT_EFFICIENCY)         0
총항해시간(TOTAL_SAILING_TIME)      0
대기시간(WAITING_TIME)             0
항만정시성(ON_TIME_PERFORMANCE)     0
dtype: int64

In [19]:
df.describe()

,항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
count,480.000000,480.000000,480.000000,480.000000
mean,19.756458,432.292521,0.138042,6.431250
std,5.754084,415.966180,0.269674,25.602098
min,6.950000,28.970000,0.000000,-322.050000
25%,16.350000,75.985000,0.000000,1.940000
50%,18.735000,327.210000,0.030000,7.645000
75%,21.427500,624.942500,0.170000,13.950000
max,47.010000,1632.540000,3.040000,135.920000


In [20]:
df_copy = df.copy()

df_copy = df_copy.rename(columns={
    '기준일자(DATE)': '기준일자', '항만효율성(PORT_EFFICIENCY)': '항만효율성', '총항해시간(TOTAL_SAILING_TIME)': '총항해시간', '대기시간(WAITING_TIME)': '대기시간', '항만정시성(ON_TIME_PERFORMANCE)': '항만정시성'})

df_copy['기준일자'] = pd.to_datetime(df_copy['기준일자'])

In [21]:
total_df = (
    df_copy[df_copy['도착항코드(DEST_PORT_CODE)'] == 'KPLI'][['기준일자', '항만효율성', '총항해시간', '대기시간', '항만정시성']]
    .rename(columns={'항만효율성': '종합항만효율성', '총항해시간': '종합총항해시간', '대기시간': '종합대기시간', '항만정시성': '종합항만정시성'}).copy()
)

route = df_copy[df_copy['도착항코드(DEST_PORT_CODE)'] != 'KPLI'].copy()

In [22]:
daily_avg = (
    route.groupby('기준일자')
    .agg(평균_항만효율성=('항만효율성', 'mean'), 평균_총항해시간=('총항해시간', 'mean'), 평균_대기시간=('대기시간', 'mean'), 평균_항만정시성=('항만정시성', 'mean')).reset_index()
)

In [23]:
compare = pd.merge(
    daily_avg, total_df, on='기준일자', how='inner'
)

compare

,기준일자,평균_항만효율성,평균_총항해시간,평균_대기시간,평균_항만정시성,종합항만효율성,종합총항해시간,종합대기시간,종합항만정시성
0,2025-05-28,18.211111,341.750000,0.123333,3.035556,18.99,239.37,0.13,2.29
1,2025-06-11,20.122500,406.270000,0.025833,13.426667,21.48,280.38,0.06,10.84
2,2025-06-11,20.122500,406.270000,0.025833,13.426667,20.45,446.38,0.00,23.58
3,2025-06-30,21.222593,510.192222,0.112593,12.230741,20.06,324.43,0.19,5.72
4,2025-06-30,21.222593,510.192222,0.112593,12.230741,20.67,319.09,0.11,9.72
5,2025-06-30,21.222593,510.192222,0.112593,12.230741,19.70,299.25,0.07,13.01
6,2025-07-08,22.956000,562.422000,0.137000,-5.026000,21.16,361.97,0.12,4.02
7,2025-07-16,18.936667,269.520000,0.150000,-4.938333,19.06,251.70,0.11,0.90
8,2025-07-29,21.402500,427.820000,0.127500,6.186875,18.63,232.41,0.15,3.09
9,2025-07-29,21.402500,427.820000,0.127500,6.186875,19.45,326.72,0.07,7.87


In [24]:
route = df[df['도착항코드(DEST_PORT_CODE)'] != 'KPLI'].copy()

route.head()

,출발항코드(DEPARTURE_PORT_CODE),출발항명(DEPARTURE_PORT_NAME),도착항코드(DEST_PORT_CODE),도착항명(DEST_PORT_NAME),도착항국가(DEST_COUNTRY),기준일자(DATE),항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
0,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-05-28,23.69,644.50,0.0,7.94
1,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,22.01,608.54,0.0,16.20
2,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,16.52,749.45,0.0,1.36
3,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,22.37,634.28,0.0,7.44
4,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,16.37,654.99,0.0,1.80


In [25]:
route_list = (route['도착항명(DEST_PORT_NAME)'].dropna().unique())
route_list

<ArrowStringArray>
[  '제벨알리항',  '안트베르펜항',    '상하이항',    '칭다오항',   '함부르크항',     '홍콩항',   '로테르담항',
   '싱가포르항',    '램차방항', '로스앤젤레스항',     '뉴욕항',    '호치민항']
Length: 12, dtype: str

In [32]:
result_list = []

for selected_route in route_list:

    filtered = route[route['도착항명(DEST_PORT_NAME)'] == selected_route].copy()

    sailing_time = filtered['총항해시간(TOTAL_SAILING_TIME)']

    mean = sailing_time.mean()
    q1 = sailing_time.quantile(0.25)
    q3 = sailing_time.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier = filtered[(sailing_time < lower_bound) | (sailing_time > upper_bound)]

    result_list.append({
        '노선': selected_route,
        '평균 총항해시간': mean,
        '전체 항해 건수': len(filtered),
        '이상치 건수': len(outlier),
        '이상치 비율': len(outlier) / len(filtered) * 100
    })

result = pd.DataFrame(result_list)

result

,노선,평균 총항해시간,전체 항해 건수,이상치 건수,이상치 비율
0,제벨알리항,647.047297,37,2,5.405405
1,안트베르펜항,1332.163913,23,1,4.347826
2,상하이항,40.304615,52,1,1.923077
3,칭다오항,39.991020,49,2,4.081633
4,함부르크항,1423.014000,15,0,0.000000
5,홍콩항,75.019355,31,0,0.000000
6,로테르담항,1335.029200,25,0,0.000000
7,싱가포르항,369.588889,54,3,5.555556
8,램차방항,267.626667,3,0,0.000000
9,로스앤젤레스항,351.327500,52,3,5.769231


In [29]:
fig = px.box(
    route,
    x='도착항명(DEST_PORT_NAME)',
    y='총항해시간(TOTAL_SAILING_TIME)',
    points='outliers',
    labels={
        '도착항명(DEST_PORT_NAME)': '도착항명',
        '총항해시간(TOTAL_SAILING_TIME)': '총항해시간'
    }
)

fig.show()